# AETHERION-CONTINUUM — Colab Image-to-3D
## Hunyuan3D-2 on free T4 GPU (~15 GB VRAM)

1. Upload sprite PNG (Runtime → Files → upload)
2. Set CHAR_KEY and IMAGE_PATH below
3. Runtime → Run all
4. Download GLB from /content/output/

local : colab : kaggle : lightning : tripo  —  five paths, one bar.

In [ ]:
# @title Configuration
CHAR_KEY = "defender"  # @param {type:"string"}
IMAGE_PATH = "/content/input.png"  # @param {type:"string"}
USE_TEXTURE = True  # @param {type:"boolean"}
FACE_LIMIT = 100000  # @param {type:"slider", min:10000, max:500000, step:10000}

import os
os.environ['AETHERION_CHAR_KEY'] = CHAR_KEY

In [ ]:
# @title GPU Check
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# @title Install Hunyuan3D-2 (~3 min)
import subprocess, sys

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q trimesh Pillow numpy huggingface_hub

import os
if not os.path.exists("Hunyuan3D-2"):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git

os.chdir("Hunyuan3D-2")
!pip install -q -r requirements.txt
!pip install -q -e .

os.chdir("hy3dgen/texgen/custom_rasterizer")
!python setup.py install
os.chdir("../../..")
os.chdir("hy3dgen/texgen/differentiable_renderer")
!python setup.py install
os.chdir("../../..")

print("Install complete.")

In [ ]:
# @title Load Model (~30s)
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.texgen import Hunyuan3DPaintPipeline

shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-2",
    subfolder="hunyuan3d-dit-v2-0",
    variant="fp16",
    device="cuda",
)
print("Shape pipeline loaded.")

if USE_TEXTURE:
    tex_pipe = Hunyuan3DPaintPipeline.from_pretrained(
        "tencent/Hunyuan3D-2",
        subfolder="hunyuan3d-paint-v2-0",
        device="cuda",
    )
    print("Texture pipeline loaded.")

In [ ]:
# @title Convert Image → 3D (~30s)
from PIL import Image
import time

img = Image.open(IMAGE_PATH).convert("RGB")
print(f"Image: {img.size}")

t0 = time.time()
mesh = shape_pipe(img, num_inference_steps=50)
print(f"Mesh: {mesh.vertices.shape[0]} vertices, {mesh.faces.shape[0]} faces ({time.time()-t0:.0f}s)")

if USE_TEXTURE:
    t0 = time.time()
    mesh = tex_pipe(mesh, img)
    print(f"Texture: applied ({time.time()-t0:.0f}s)")

In [ ]:
# @title Export GLB
import os
os.makedirs("/content/output", exist_ok=True)

out_path = f"/content/output/{CHAR_KEY}.glb"
mesh.export(out_path)
print(f"Exported: {out_path}")

from google.colab import files
files.download(out_path)